# Exploratory Data Analysis: NYT Coverage of China and Russia (2020–2024)

This notebook produces the data-cleaning pipeline and the macro-structural figures used in **Section 4** and the corresponding appendices of the thesis. All analysis prose has been removed; full discussion appears in the manuscript. Each code block below is labeled with the figure it produces and its location in the paper.

## Contents

| Section | Output | Paper location |
| --- | --- | --- |
| 1. Data Cleaning Pipeline | `NYT_Master_Cleaned_2020-2024.csv` | §3.2 (Data) |
| 2.1 Coverage Volume Over Time | Figure 1 | §4.1 |
| 2.2 Coverage Volume vs. Total NYT Output | Figure 2 | §4.1 |
| 2.3 Section Distribution by Article Category | Figure 3 | §4.2 |
| 3.1 Top Keywords by Section × Category | Figure A1 | Appendix A |
| 3.2 Temporal Evolution of Issue Vocabularies | Figure 4 | §4.3 |
| 4.1 Leadership Mentions (NER, PERSON) | Figure B1 | Appendix B |
| 4.2 Geographic Focus (NER, GPE) | Figure B2 | Appendix B |

## Inputs and outputs

- **Inputs**: `China_Metadata_Merged_2020-2024.csv`, `Russia_Metadata_Merged_2020-2024.csv` (produced upstream by the scraping pipeline; not included here).
- **Output**: `NYT_Master_Cleaned_2020-2024.csv`, used by Section 1 and read directly by the NER cells in Section 4.

## Dependencies

`pandas`, `matplotlib`, `seaborn`, `numpy`, `nltk` (with `stopwords`), `spacy` (with `en_core_web_sm`).

## Note on outputs and reproducibility

Cell outputs have been stripped to keep the file lightweight; running all cells reproduces the figures end-to-end. Two analyses involve stochastic components: the sentence-level disaggregation in §5.4 draws a random 20,000-sentence sample, and the Word2Vec models in §7 are trained with multi-core parallelism. Re-running these cells produces results within ±0.01 of the thesis numbers but not exact bitwise replicas. The figures and exact statistics reported in the manuscript correspond to a specific run preserved in the author's working directory.


## 1. Data Cleaning Pipeline

Merges the China-keyed and Russia-keyed metadata exports into a single master table, coalesces duplicate columns, fills missing flags, parses dates, assigns each article an `article_category` (China Only, Russia Only, Both, Other), and writes the cleaned dataset to disk. All downstream sections read from `NYT_Master_Cleaned_2020-2024.csv`.


**1.1 Merge China and Russia metadata exports.**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the merged metadata files for China and Russia
df_china = pd.read_csv('China_Metadata_Merged_2020-2024.csv')
df_russia = pd.read_csv('Russia_Metadata_Merged_2020-2024.csv')

df_master = pd.merge(
    df_china, 
    df_russia, 
    on='web_url', 
    how='outer', 
    suffixes=('_ch', '_ru')
)

**1.2 Coalesce duplicate metadata columns** (prefer China-side, fall back to Russia-side).

In [ ]:
metadata_cols = [
    'headline', 'pub_date', 'news_desk', 
    'document_type', 'snippet', 'abstract', 'lead_paragraph'
]

for col in metadata_cols:
    col_ch = f"{col}_ch"
    col_ru = f"{col}_ru"
    
    # Check column existence to avoid errors
    if col_ch in df_master.columns and col_ru in df_master.columns:
        # Prefer China-side data; fall back to Russia-side when missing
        df_master[col] = df_master[col_ch].combine_first(df_master[col_ru])
        # Drop redundant suffixed columns to keep the table tidy
        df_master.drop(columns=[col_ch, col_ru], inplace=True)

**1.3 Fill missing rival-flag indicators.**

In [ ]:
if 'is_china_related' in df_master.columns:
    df_master['is_china_related'] = df_master['is_china_related'].fillna(False)

if 'is_russia_related' in df_master.columns:
    df_master['is_russia_related'] = df_master['is_russia_related'].fillna(False)

**1.4 Parse publication date as datetime.**

In [ ]:
df_master['pub_date'] = pd.to_datetime(df_master['pub_date'])

**1.5 Inspect news-desk distribution** (used in §4.2).

In [ ]:
#give me the types of news_desk
df_master['news_desk'].value_counts()

**1.6 Assign article category** (China Only, Russia Only, Both, Other).

In [ ]:
def get_category(row):
    c = row.get('is_china_related', False)
    r = row.get('is_russia_related', False)
    
    if c and r:
        return 'Both'        # Articles referencing both China and Russia (Nexus)
    elif c:
        return 'China Only'  # China-only articles
    elif r:
        return 'Russia Only' # Russia-only articles
    else:
        return 'Other'       # Should not occur in this corpus

df_master['article_category'] = df_master.apply(get_category, axis=1)

**1.7 Save the cleaned master dataset.**

In [ ]:
output_filename = 'NYT_Master_Cleaned_2020-2024.csv'
df_master.to_csv(output_filename, index=False)

## 2. Volume and Topical Distribution

Three figures establish the macro-temporal architecture and institutional routing of NYT coverage of China versus Russia.


### 2.1 Figure 1 — Monthly Article Volume: China vs. Russia (§4.1)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates

# ==========================================
# 0. Load Data
# ==========================================
# Read the cleaned master dataset generated in the previous step
df = pd.read_csv('NYT_Master_Cleaned_2020-2024.csv')

# Ensure publication date is in datetime format
df['pub_date'] = pd.to_datetime(df['pub_date'])

# Create a monthly period column for aggregation
df['year_month'] = df['pub_date'].dt.to_period('M')

print(f"Data loaded successfully. Total articles: {len(df)}")

# ==========================================
# 1. Calculate Broad Aggregations
# ==========================================
# We need to calculate three metrics per month:
# 1. Total Articles: All relevant articles in the dataset
# 2. China Related: Includes 'China Only' AND 'Both' (Nexus)
# 3. Russia Related: Includes 'Russia Only' AND 'Both' (Nexus)

# Group by month first
monthly_groups = df.groupby('year_month')

# Aggregate counts
monthly_stats = pd.DataFrame({
    'Total Articles': monthly_groups.size(),
    
    # Filter for China-related categories and count
    'China Related': df[df['article_category'].isin(['China Only', 'Both'])].groupby('year_month').size(),
    
    # Filter for Russia-related categories and count
    'Russia Related': df[df['article_category'].isin(['Russia Only', 'Both'])].groupby('year_month').size()
})

# Fill NaN values with 0 (for months with no coverage in a specific category)
monthly_stats = monthly_stats.fillna(0)

# Convert index to timestamp for plotting compatibility
monthly_stats.index = monthly_stats.index.to_timestamp()

# ==========================================
# 2. Plot General Trends (The "Panorama")
# ==========================================
plt.figure(figsize=(14, 7))
sns.set_style("whitegrid") # Set background grid style

# Plot 1: Total Volume (Area Chart background)
# Uses a light gray fill to show the overall capacity/attention of the outlet
plt.fill_between(monthly_stats.index, monthly_stats['Total Articles'], color='gray', alpha=0.1, label='Total Coverage Volume')
plt.plot(monthly_stats.index, monthly_stats['Total Articles'], color='gray', linestyle=':', alpha=0.5)

# Plot 2: China Related (Blue Line)
# Represents broad attention to China (both bilateral and multilateral)
plt.plot(monthly_stats.index, monthly_stats['China Related'], label='China-Related', color='#005a9c', linewidth=2.5)

# Plot 3: Russia Related (Red Line)
# Represents broad attention to Russia
plt.plot(monthly_stats.index, monthly_stats['Russia Related'], label='Russia-Related', color='#d62728', linewidth=2.5)

# ==========================================
# 3. Contextual Event Anchors (CLEAN: numbered markers + event key)
# ==========================================
events = [
    # --- China / US-China / Taiwan ---
    ("2020-01-23", "Wuhan lockdown begins", "China Related"),
    ("2020-06-30", "Hong Kong National Security Law", "China Related"),
    ("2021-03-22", "US/EU sanction China over Xinjiang", "China Related"),
    ("2022-08-02", "Pelosi visits Taiwan", "China Related"),
    ("2022-10-16", "CCP Party Congress: Xi consolidates power", "China Related"),
    ("2023-02-04", "Spy balloon incident", "China Related"),
    ("2023-11-15", "APEC: Xi meets Biden (SF)", "China Related"),
    ("2024-01-13", "Taiwan election", "China Related"),

    # --- Russia / Ukraine ---
    ("2022-02-24", "Russia invades Ukraine", "Russia Related"),
    ("2022-09-26", "Nord Stream explosions", "Russia Related"),
    ("2022-09-30", "Annexation claims (4 regions)", "Russia Related"),
    ("2023-06-24", "Wagner mutiny", "Russia Related"),
    ("2023-08-23", "Prigozhin dies", "Russia Related"),
    ("2024-03-17", "Putin re-elected", "Russia Related"),
]

ax = plt.gca()

# Make room on the right for the event key
plt.subplots_adjust(right=0.78)

# Optional: color-code markers by series (matches your line colors)
series_color = {
    "China Related": "#005a9c",
    "Russia Related": "#d62728",
}

event_key_lines = []
n = 1

for (date_str, label, series_name) in events:
    date_obj = pd.to_datetime(date_str)

    idx = monthly_stats.index.asof(date_obj)
    if pd.isna(idx) or idx not in monthly_stats.index:
        continue

    x = idx
    y = float(monthly_stats.loc[idx, series_name])

    # marker + number
    ax.scatter(
        [x], [y],
        s=55,
        color=series_color.get(series_name, "black"),
        zorder=9
    )
    ax.text(
        x, y,
        str(n),
        ha="center", va="center",
        fontsize=8, fontweight="bold",
        color="white",
        zorder=10
    )

    # event key line
    tag = "C" if series_name == "China Related" else "R"
    event_key_lines.append(f"{n:>2}. [{tag}] {date_obj.strftime('%Y-%m')} — {label}")
    n += 1

# Draw the event key box on the right (outside plotting area)
event_key = "Event Key\n" + "\n".join(event_key_lines)
ax.text(
    1.01, 0.98,
    event_key,
    transform=ax.transAxes,
    va="top", ha="left",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="none", alpha=0.92)
)
# ==========================================
# 4. Chart Formatting
# ==========================================
plt.title('General Trend: NYT Coverage of China vs. Russia (2020-2024)', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Number of Articles per Month', fontsize=12)
plt.legend(loc='upper left', fontsize=11, frameon=True)

# Format X-axis to show ticks every 6 months for readability
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

# ==========================================
# 5. Print Summary Statistics
# ==========================================
print("--- General Statistics (2020-2024) ---")
print(f"Total Article Volume: {monthly_stats['Total Articles'].sum()}")
print(f"Total China-Related: {int(monthly_stats['China Related'].sum())}")
print(f"Total Russia-Related: {int(monthly_stats['Russia Related'].sum())}")

# Calculate Correlation
# This measures if coverage of China and Russia moves together (positive) or inversely (negative)
corr = monthly_stats['China Related'].corr(monthly_stats['Russia Related'])
print(f"\nMonthly Correlation (China vs. Russia): {corr:.2f}")

if corr > 0.5:
    print("Interpretation: Strong positive correlation. Crises tend to involve both nations or occur simultaneously.")
elif corr < -0.2:
    print("Interpretation: Negative correlation. Evidence of 'attention displacement' (zero-sum media attention).")
else:
    print("Interpretation: Low correlation. Coverage is driven by independent events rather than a unified narrative.")

### 2.2 Figure 2 — Coverage vs. Total Dataset Volume (§4.1)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ==========================================
# 0. Data Prep
# ==========================================
# Use existing dataframe
plot_df = df.copy() # Or pd.read_csv('NYT_Master_Cleaned_2020-2024.csv')

# Ensure datetime
plot_df['pub_date'] = pd.to_datetime(plot_df['pub_date'])
plot_df['month'] = plot_df['pub_date'].dt.to_period('M')

# ==========================================
# 1. Calculate Monthly Volumes
# ==========================================
# Total Volume in YOUR dataset (Proxy for "Geopolitical News Hole")
total_vol = plot_df.groupby('month').size()

# China Related Volume (China Only + Both)
china_vol = plot_df[plot_df['article_category'].isin(['China Only', 'Both'])].groupby('month').size()

# Russia Related Volume (Russia Only + Both)
russia_vol = plot_df[plot_df['article_category'].isin(['Russia Only', 'Both'])].groupby('month').size()

both_vol = plot_df[plot_df['article_category'].isin(['Both'])].groupby('month').size()

# Combine into a clean DataFrame for stats
stats_df = pd.DataFrame({
    'Total_Dataset_Vol': total_vol,
    'China_Vol': china_vol,
    'Russia_Vol': russia_vol,
    'Both_Vol': both_vol
}).fillna(0)

# ==========================================
# 2. Statistical Test: Correlation Matrix
# ==========================================
# We calculate Pearson Correlation.
# If r is close to 1: They move together (Dependent).
# If r is close to 0: No relationship (Independent - What you want!).
corr_matrix = stats_df.corr()

print("--- Correlation Matrix (Pearson) ---")
print(corr_matrix)

# Extract specific correlations for the thesis text
china_total_corr = corr_matrix.loc['China_Vol', 'Total_Dataset_Vol']
russia_total_corr = corr_matrix.loc['Russia_Vol', 'Total_Dataset_Vol']

# ==========================================
# 3. Visualization: Dual-Axis Chart
# ==========================================
fig, ax1 = plt.subplots(figsize=(14, 7))
sns.set_style("white")

# Plot 1 (Background): Total Dataset Volume (The "News Hole" Proxy)
# Using Bar chart on the primary Y-axis (Left)
color_total = 'lightgray'
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Total Dataset Volume (Background)', color='gray', fontsize=12)
stats_df['Total_Dataset_Vol'].plot(kind='bar', ax=ax1, color=color_total, width=1.0, alpha=0.3, label='Total Collected Articles')
ax1.tick_params(axis='y', labelcolor='gray')
# Remove too many x-labels
ax1.set_xticks(range(0, len(stats_df), 6))
ax1.set_xticklabels([x.strftime('%Y-%m') for x in stats_df.index.to_timestamp()[::6]])

# Create a twin Axes sharing the xaxis for the Lines
ax2 = ax1.twinx() 

# Plot 2 (Foreground): Specific Country Trends
# Using Line chart on the secondary Y-axis (Right)
color_china = '#1f77b4' # Blue
color_russia = '#d62728' # Red

# Note: We plot against a range to match the bar chart x-axis
x_indices = range(len(stats_df))
ax2.plot(x_indices, stats_df['China_Vol'], color=color_china, linewidth=2.5, label='China Coverage')
ax2.plot(x_indices, stats_df['Russia_Vol'], color=color_russia, linewidth=2.5, label='Russia Coverage')
# Both (Nexus) line — green dashed
ax2.plot(x_indices, stats_df['Both_Vol'], label='Both (Nexus)', color='#2ca02c',
         linewidth=2.5, linestyle='--')

ax2.set_ylabel('Specific Country Coverage', color='black', fontsize=12)
ax2.tick_params(axis='y', labelcolor='black')

# Add Legend (Manually combining both axes)
lines, labels = ax2.get_legend_handles_labels()
bars, bar_labels = ax1.get_legend_handles_labels()
ax2.legend(bars + lines, bar_labels + labels, loc='upper left', frameon=True)



plt.title('Macro Trend Analysis: Specific Coverage vs. Total Dataset Volume', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# ==========================================
# 4. Interpretation Logic (Output for you)
# ==========================================
print(f"\n--- Interpretation for Thesis ---")
print(f"China vs. Total Correlation: {china_total_corr:.2f}")
print(f"Russia vs. Total Correlation: {russia_total_corr:.2f}")

if china_total_corr < 0.5:
    print("RESULT: China coverage shows LOW correlation with total volume.")
    print("MEANING: China coverage is independent and event-driven, not a byproduct of general NYT output.")
else:
    print("RESULT: China coverage moves with total volume.")
    
if russia_total_corr < 0.5:
    print("RESULT: Russia coverage shows LOW correlation with total volume.")
    print("MEANING: Russia coverage is highly specific to its own crises (e.g., Ukraine War).")

### 2.3 Figure 3 — Section Distribution by Article Category (§4.2)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


plot_df = df.copy()

# ==========================================
# 1. Map News Desk to Broader Categories
# ==========================================
# The raw 'news_desk' column has many specific names (e.g., "SundayBusiness", "Foreign").
# We need to group them into 5-6 meaningful analytical categories.

def map_section(desk):
    # Convert to string and lowercase to avoid errors
    desk = str(desk).lower()
    
    # Define mapping logic
    if 'business' in desk or 'financial' in desk or 'dealbook' in desk or 'economy' in desk:
        return 'Business/Econ'
    elif 'tech' in desk or 'science' in desk or 'climate' in desk:
        return 'Tech/Science'
    elif 'world' in desk or 'international' in desk or 'foreign' in desk:
        return 'World/Security'
    elif 'politics' in desk or 'washington' in desk:
        return 'U.S. Politics'
    elif 'opinion' in desk or 'editorial' in desk or 'letter' in desk or 'op-ed' in desk:
        return 'Opinion'
    else:
        return 'Other' # Includes Arts, Style, Sports, etc.

# Apply the mapping
plot_df['section_group'] = plot_df['news_desk'].apply(map_section)

# ==========================================
# 2. Calculate Proportions
# ==========================================
# We want to know: For each Topic (China/Russia/Both), what is the breakdown of sections?

# Group by Topic Category and Section Group
section_counts = plot_df.groupby(['article_category', 'section_group']).size().unstack(fill_value=0)

# Convert counts to percentages (Normalize along the row)
# This makes the bars comparable even if total article counts differ
section_pct = section_counts.div(section_counts.sum(axis=1), axis=0) * 100

# ==========================================
# 3. Plot Stacked Bar Chart
# ==========================================
plt.figure(figsize=(12, 8))
sns.set_style("white")

# Plot using pandas built-in plotting for easy stacking
# 'Spectral' colormap gives distinct colors for sections
ax = section_pct.plot(kind='bar', stacked=True, figsize=(12, 7), 
                      colormap='Spectral', edgecolor='black', width=0.7)

# ==========================================
# 4. Formatting & Labels
# ==========================================
plt.title('Distribution of Coverage by Section & Topic (2020-2024)', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Percentage of Articles (%)', fontsize=12)
plt.xlabel('Topic Category', fontsize=12)
plt.xticks(rotation=0) # Keep x-axis labels horizontal

# Move legend outside the plot area
plt.legend(title='News Section', bbox_to_anchor=(1.05, 1), loc='upper left')

# Add percentage labels on the bars
for c in ax.containers:
    # Only label segments that are big enough (> 5%) to avoid clutter
    labels = [f'{v.get_height():.1f}%' if v.get_height() > 5 else '' for v in c]
    ax.bar_label(c, labels=labels, label_type='center', fontsize=10, color='black', weight='bold')

plt.tight_layout()
plt.show()

# ==========================================
# 5. Print Summary Table
# ==========================================
print("--- Section Distribution Table (%) ---")
print(section_pct.round(2))

## 3. Issue Vocabularies

Static and longitudinal keyword analyses across the four article-category × section subsets (China × Business, China × World, Russia × Business, Russia × World).


### 3.1 Figure A1 — Top Keywords in Headlines and Leads (Appendix A)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import nltk
from nltk.corpus import stopwords

# ==========================================
# 0. Setup & Data Loading
# ==========================================
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

plot_df = df.copy()

# ==========================================
# 1. Aggressive Noise Removal List
# ==========================================
base_stopwords = set(stopwords.words('english'))

aggressive_noise = {
    'one', 'two', 'three', 'four', 'five', 'first', 'second', 'many', 'much', 'several','billion','made',
    'would', 'could', 'should', 'may', 'might', 'must', 'can', 'will', 'also', 'even', 'still','real','biggest','largest', 'make',
    'country', 'countries', 'nation', 'nations', 'world', 'global', 'city', 'cities',
    'government', 'governments', 'official', 'officials', 'leader', 'leaders',
    'people', 'public', 'state', 'states','days','south','back',
    'said', 'says', 'told', 'reported', 'mr', 'ms', 'year', 'years', 'time', 'day', 'week', 'month', 'like',
    'today', 'yesterday', 'last', 'next', 'new', 'old', 'high', 'low', 'big', 'major','tuesday', 'wednesday', 'thursday', 'friday', 'monday','top', 'pro','cases',
    'company', 'companies', 'business', 'businesses', 'economy', 'economic', 'industry', 'market',
    'china', 'chinese', 'russia', 'russian', 'beijing', 'moscow', 'putin', 'vladimir', 'xi', 'jinping',
    'biden', 'trump', 'usa', 'united', 'american', 'americans','president',
}

all_stopwords = base_stopwords.union(aggressive_noise)

def clean_and_tokenize(text_series):
    text = " ".join(text_series.fillna('').astype(str).tolist()).lower()
    text = text.replace('hong kong', 'hongkong')
    text = text.replace('united states', 'usa')
    text = text.replace('social media', 'social_media')
    text = text.replace('coronavirus', 'covid')
    text = text.replace('pandemic', 'covid')
    text = text.replace('technology', 'tech')
    
    tokens = re.findall(r'\b[a-z]{3,}\b', text)
    filtered_tokens = [t for t in tokens if t not in all_stopwords]
    return filtered_tokens

plot_df['combined_text'] = plot_df['headline'].fillna('') + " " + plot_df['lead_paragraph'].fillna('')

# ==========================================
# 2. Extract Keywords
# ==========================================
def map_section(desk):
    desk = str(desk).lower()
    if 'business' in desk or 'financial' in desk or 'dealbook' in desk or 'economy' in desk:
        return 'Business/Econ'
    elif 'world' in desk or 'international' in desk or 'foreign' in desk:
        return 'World/Security'
    else:
        return 'Other'

plot_df['section_group'] = plot_df['news_desk'].apply(map_section)

# Create Subsets (now 4)
subset_china_biz    = plot_df[(plot_df['article_category'] == 'China Only')  & (plot_df['section_group'] == 'Business/Econ')]
subset_china_world  = plot_df[(plot_df['article_category'] == 'China Only')  & (plot_df['section_group'] == 'World/Security')]
subset_russia_biz   = plot_df[(plot_df['article_category'] == 'Russia Only') & (plot_df['section_group'] == 'Business/Econ')]
subset_russia_world = plot_df[(plot_df['article_category'] == 'Russia Only') & (plot_df['section_group'] == 'World/Security')]

count_china_biz    = Counter(clean_and_tokenize(subset_china_biz['combined_text'])).most_common(20)
count_china_world  = Counter(clean_and_tokenize(subset_china_world['combined_text'])).most_common(20)
count_russia_biz   = Counter(clean_and_tokenize(subset_russia_biz['combined_text'])).most_common(20)
count_russia_world = Counter(clean_and_tokenize(subset_russia_world['combined_text'])).most_common(20)

# Sanity check: Russia Biz sample size often small
print(f"Russia Business/Econ articles: {len(subset_russia_biz)}")

# ==========================================
# 3. Visualization (2x2)
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
sns.set_style("whitegrid")

def plot_bar(data, ax, title, color):
    if not data:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes, fontsize=14)
        ax.set_title(title, fontsize=14, fontweight='bold')
        return
    words = [x[0] for x in data][::-1]
    counts = [x[1] for x in data][::-1]
    ax.barh(words, counts, color=color, edgecolor='black', alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Frequency', fontsize=12)
    ax.tick_params(axis='y', labelsize=11)

plot_bar(count_china_biz,    axes[0, 0], 'China (Business Section)',  '#ffcc00')  # Gold
plot_bar(count_china_world,  axes[0, 1], 'China (World Section)',     '#1f77b4')  # Blue
plot_bar(count_russia_biz,   axes[1, 0], 'Russia (Business Section)', '#ff7f0e')  # Orange
plot_bar(count_russia_world, axes[1, 1], 'Russia (World Section)',    '#d62728')  # Red

plt.suptitle('Top Keywords in NYT Coverage: Specific Issues (2020-2024)', fontsize=18, y=1.00)
plt.tight_layout()
plt.show()

### 3.2 Figure 4 — Temporal Evolution of Top Keywords, 2020–2024 (§4.3)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import nltk
from nltk.corpus import stopwords

# ==========================================
# 0. Setup & Helper Functions
# ==========================================
plot_df = df.copy()

plot_df['pub_date'] = pd.to_datetime(plot_df['pub_date'])
plot_df['year'] = plot_df['pub_date'].dt.year

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
    
base_stopwords = set(stopwords.words('english'))
custom_noise = {
    'one', 'two', 'three', 'four', 'five', 'first', 'second', 'many', 'much', 'several','billion', 'biggest',
    'would', 'could', 'should', 'may', 'might', 'must', 'can', 'will', 'also', 'even', 'still',
    'country', 'countries', 'nation', 'nations', 'world', 'global', 'city', 'cities',
    'government', 'governments', 'official', 'officials', 'leader', 'leaders',
    'people', 'public', 'state', 'states','days','made','top','south','largest','real',
    'said', 'says', 'told', 'reported', 'mr', 'ms', 'year', 'years', 'time', 'day', 'week', 'month', 'like',
    'today', 'yesterday', 'last', 'next', 'new', 'old', 'high', 'low', 'big', 'major','tuesday', 'wednesday', 'thursday', 'friday', 'monday', 'said', 'data',
    'company', 'companies', 'business', 'businesses', 'economy', 'economic', 'industry', 'market',
    'china', 'chinese', 'russia', 'russian', 'beijing', 'moscow', 'putin', 'vladimir', 'xi', 'jinping',
    'biden', 'trump', 'usa', 'united', 'american', 'americans'
}
all_stopwords = base_stopwords.union(custom_noise)

def get_keywords_by_year(subset_df):
    yearly_counters = {}
    
    for year in sorted(subset_df['year'].unique()):
        year_data = subset_df[subset_df['year'] == year]
        
        text = " ".join(year_data['headline'].fillna('') + " " + year_data['lead_paragraph'].fillna('')).lower()
        
        text = text.replace('hong kong', 'hongkong')
        text = text.replace('united states', 'usa')
        text = text.replace('social media', 'social_media')
        text = text.replace('coronavirus', 'covid')
        text = text.replace('pandemic', 'covid')
        text = text.replace('communist party', 'ccp')
        
        tokens = re.findall(r'\b[a-z]{3,}\b', text)
        tokens = [t for t in tokens if t not in all_stopwords]
        
        yearly_counters[year] = Counter(tokens)
    
    if not yearly_counters:
        return pd.DataFrame()
    
    df_keywords = pd.DataFrame(yearly_counters).fillna(0).astype(int)
    
    total_counts = df_keywords.sum(axis=1)
    top_keywords = total_counts.sort_values(ascending=False).head(15).index
    
    return df_keywords.loc[top_keywords]

# ==========================================
# 1. Prepare Data for Heatmaps
# ==========================================
def map_section(desk):
    desk = str(desk).lower()
    if 'business' in desk or 'financial' in desk or 'dealbook' in desk or 'economy' in desk:
        return 'Business/Econ'
    elif 'world' in desk or 'international' in desk or 'foreign' in desk:
        return 'World/Security'
    else:
        return 'Other'

plot_df['section_group'] = plot_df['news_desk'].apply(map_section)

# Create Subsets (now 4)
subset_china_biz    = plot_df[(plot_df['article_category'] == 'China Only')  & (plot_df['section_group'] == 'Business/Econ')]
subset_china_world  = plot_df[(plot_df['article_category'] == 'China Only')  & (plot_df['section_group'] == 'World/Security')]
subset_russia_biz   = plot_df[(plot_df['article_category'] == 'Russia Only') & (plot_df['section_group'] == 'Business/Econ')]
subset_russia_world = plot_df[(plot_df['article_category'] == 'Russia Only') & (plot_df['section_group'] == 'World/Security')]

print(f"Russia Business/Econ articles: {len(subset_russia_biz)}")

matrix_china_biz    = get_keywords_by_year(subset_china_biz)
matrix_china_world  = get_keywords_by_year(subset_china_world)
matrix_russia_biz   = get_keywords_by_year(subset_russia_biz)
matrix_russia_world = get_keywords_by_year(subset_russia_world)

# ==========================================
# 2. Plot Heatmaps (2x2)
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

def plot_heatmap(matrix, ax, title, cmap):
    if matrix.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes, fontsize=14)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])
        return
    # Normalize by column (Year): shows relative importance within that year
    matrix_norm = matrix.div(matrix.sum(axis=0), axis=1)
    
    sns.heatmap(matrix_norm, ax=ax, cmap=cmap, annot=False, cbar=False, linewidths=.5)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel('')
    ax.set_xlabel('Year')
    ax.tick_params(axis='y', labelsize=11)
    ax.tick_params(axis='x', rotation=45)

plot_heatmap(matrix_china_biz,    axes[0, 0], 'Evolution of China Keywords (Business)',  'YlOrBr')
plot_heatmap(matrix_china_world,  axes[0, 1], 'Evolution of China Keywords (World)',     'Blues')
plot_heatmap(matrix_russia_biz,   axes[1, 0], 'Evolution of Russia Keywords (Business)', 'Oranges')
plot_heatmap(matrix_russia_world, axes[1, 1], 'Evolution of Russia Keywords (World)',    'Reds')

plt.suptitle('Temporal Evolution of Media Frames: Top Keywords (2020-2024)', fontsize=18, y=1.00)
plt.tight_layout()
plt.show()

## 4. Named Entity Analysis (Appendix B)

Entity-level distributions for leadership (PERSON) and geographic anchoring (GPE), extracted from headlines and leads via spaCy `en_core_web_sm`. Results support the §4.4 summary and are reported in full in Appendix B.


### 4.1 Figure B1 — Most Frequent Leaders in China vs. Russia Coverage

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
from collections import Counter

# ==========================================
# 0. Setup & Data Loading
# ==========================================
# Load spaCy English model (run `python -m spacy download en_core_web_sm` first if needed)
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Downloading spaCy model...")
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

# Use the in-memory DataFrame
# df_ner = df.copy()  # If already loaded earlier in the session
# Otherwise load from disk:
df_ner = pd.read_csv('NYT_Master_Cleaned_2020-2024.csv')

# Build the NER input text from headline and lead paragraph
df_ner['text_for_ner'] = df_ner['headline'].fillna('') + " " + df_ner['lead_paragraph'].fillna('')

# ==========================================
# 1. NER Function
# ==========================================
def extract_entities(text_series):
    """
    Extract PERSON entities from a text series.
    """
    people = []
    # Use nlp.pipe for batched processing
    for doc in nlp.pipe(text_series.astype(str), batch_size=50, disable=["parser"]):
        for ent in doc.ents:
            if ent.label_ == "PERSON":
                people.append(ent.text)
    return people

# ==========================================
# 2. Extract Entities for China vs. Russia
# ==========================================
print("Extracting entities for China articles... (this may take a minute)")
china_text = df_ner[df_ner['article_category'] == 'China Only']['text_for_ner']
china_people = extract_entities(china_text)

print("Extracting entities for Russia articles...")
russia_text = df_ner[df_ner['article_category'] == 'Russia Only']['text_for_ner']
russia_people = extract_entities(russia_text)

# ==========================================
# 3. Clean & Count
# ==========================================
# Tokens to ignore (reporter names or misclassified terms)
ignore_list = {'Xi', 'Putin', 'Biden', 'Trump', 'Briefing', 'Correspondent', 'Reporter'} 

def clean_counts(entity_list):
    # Normalize surface-form variants to canonical names
    cleaned = []
    for name in entity_list:
        name_lower = name.lower()
        if 'xi' in name_lower or 'jinping' in name_lower:
            cleaned.append('Xi Jinping')
        elif 'putin' in name_lower or 'vladimir' in name_lower:
            cleaned.append('Vladimir Putin')
        elif 'biden' in name_lower:
            cleaned.append('Joe Biden')
        elif 'trump' in name_lower:
            cleaned.append('Donald Trump')
        elif 'zelensky' in name_lower:
            cleaned.append('Volodymyr Zelensky')
        elif 'navalny' in name_lower:
            cleaned.append('Alexei Navalny')
        elif 'blinken' in name_lower:
            cleaned.append('Antony Blinken')
        else:
             # Keep other high-frequency names; inspect before plotting
             cleaned.append(name)
    return Counter(cleaned)

china_counts = clean_counts(china_people)
russia_counts = clean_counts(russia_people)

# ==========================================
# 4. Visualization
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.set_style("whitegrid")

# Helper plot function
def plot_top_people(counter, ax, title, color):
    # Get Top 10
    top_10 = counter.most_common(10)
    names = [x[0] for x in top_10][::-1]
    counts = [x[1] for x in top_10][::-1]
    
    ax.barh(names, counts, color=color, edgecolor='black', alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Mentions in Headlines/Leads', fontsize=12)

plot_top_people(china_counts, axes[0], 'Top Figures in China Coverage', '#1f77b4')
plot_top_people(russia_counts, axes[1], 'Top Figures in Russia Coverage', '#d62728')

plt.suptitle('Leadership Portrayals: Who Drives the Narrative?', fontsize=16, y=1.05)
plt.tight_layout()
plt.show()

### 4.2 Figure B2 — Geographic Focus in China vs. Russia Coverage

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
from collections import Counter

# ==========================================
# 0. Setup
# ==========================================
# Use existing dataframe
plot_df = df.copy()

# Load Spacy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

# Define Text
plot_df['text_for_ner'] = plot_df['headline'].fillna('') + " " + plot_df['lead_paragraph'].fillna('')

# ==========================================
# 1. Extract Locations (GPE)
# ==========================================
def extract_locations(text_series):
    locations = []
    # Batch processing for speed
    for doc in nlp.pipe(text_series.astype(str), batch_size=50, disable=["parser"]):
        for ent in doc.ents:
            if ent.label_ == "GPE": # GPE = Countries, Cities, States
                locations.append(ent.text)
    return locations

print("Extracting locations for China coverage...")
china_locs = extract_locations(plot_df[plot_df['article_category'] == 'China Only']['text_for_ner'])

print("Extracting locations for Russia coverage...")
russia_locs = extract_locations(plot_df[plot_df['article_category'] == 'Russia Only']['text_for_ner'])

# ==========================================
# 2. Clean & Count
# ==========================================
# Clean up variations (e.g., "U.S.", "United States" -> "USA")
def clean_locs(loc_list):
    cleaned = []
    for loc in loc_list:
        l = loc.lower()
        # --- Aggressive consolidation rules ---
        if l in ['u.s.', 'united states', 'america', 'usa', 'us', 'the united states', 'washington']:
            cleaned.append('USA (Context)')  # Treat Washington as policy-arena context
        elif l in ['china', 'chinese', 'beijing']:  # Merge Beijing into China
            cleaned.append('China')
        elif l in ['russia', 'russian', 'moscow']:  # Merge Moscow into Russia
            cleaned.append('Russia')
        elif l in ['ukraine', 'ukrainian', 'kyiv', 'kiev']:  # Merge Kyiv into Ukraine
            cleaned.append('Ukraine')
        # ... others kept as-is ...
        elif l in ['taiwan']:
            cleaned.append('Taiwan')
        elif l in ['hong kong', 'hong kong’s']:
            cleaned.append('Hong Kong')
        else:
            cleaned.append(loc)  # Keep as-is
            
    # Remove the countries themselves (China/Russia/USA) to focus on CITIES/REGIONS if you want
    # But usually keeping "Washington" vs "Beijing" is the key comparison.
    # Let's remove the generic country names to focus on specific centers of gravity
    # Optional: comment out the next line if you want to see "China" vs "USA"
    # cleaned = [x for x in cleaned if x not in ['China', 'Russia', 'USA', 'Ukraine']]
    
    return Counter(cleaned)

china_cnt = clean_locs(china_locs)
russia_cnt = clean_locs(russia_locs)

# ==========================================
# 3. Visualization
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.set_style("whitegrid")

def plot_top_locs(counter, ax, title, color):
    top_10 = counter.most_common(10)
    names = [x[0] for x in top_10][::-1]
    counts = [x[1] for x in top_10][::-1]
    ax.barh(names, counts, color=color, edgecolor='black', alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Mentions in Headlines/Leads')

plot_top_locs(china_cnt, axes[0], 'Geographic Focus: China Coverage', '#1f77b4')
plot_top_locs(russia_cnt, axes[1], 'Geographic Focus: Russia Coverage', '#d62728')

plt.suptitle('Where is the Story? Geographic "Domestication" Analysis', fontsize=16, y=1.05)
plt.tight_layout()
plt.show()